# UDPLag — Feature Selection & Finetune Dataset
Dataset **balanceado 50/50 antes do split**, replicando o comportamento original.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

## 1. Carregamento e pré-processamento

In [ ]:
CSV_PATH    = "/content/UDPLag.csv"
RANDOM_SEED = 42

df = pd.read_csv(CSV_PATH, low_memory=False)
df.columns = df.columns.str.strip()
df = df[df["Label"].isin(["UDPLag", "BENIGN"])].copy()
df["Label"] = df["Label"].astype(str).str.strip()

print(f"UDPLag: {len(df[df['Label'] == 'UDPLag'])}")
print(f"BENIGN: {len(df[df['Label'] == 'BENIGN'])}")

# Label binário
df["Label_enc"] = (df["Label"] == "UDPLag").astype(int)

## 2. Limpeza de valores infinitos e NaN

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "Label_enc"]

for col in numeric_cols:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].fillna(df[col].median())

## 3. ✅ Balanceamento 50/50 ANTES do split
> Esta é a correção principal em relação ao script anterior.
> O dataset é balanceado aqui para garantir igual representação de classes no treino e no teste.

In [ ]:
n_min = df["Label_enc"].value_counts().min()

df_balanced = (
    df.groupby("Label_enc", group_keys=False)
      .apply(lambda x: x.sample(n=n_min, random_state=RANDOM_SEED))
      .sample(frac=1, random_state=RANDOM_SEED)   # embaralha
      .reset_index(drop=True)
)

print(f"Dataset balanceado: {len(df_balanced)} amostras")
print(df_balanced["Label"].value_counts())

## 4. Split treino / teste (80/20)

In [ ]:
train_df, test_df = train_test_split(
    df_balanced,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=df_balanced["Label_enc"]
)

print("Treino:", train_df.shape, " | Teste:", test_df.shape)
print("\nDistribuição treino:")
print(train_df["Label"].value_counts())
print("\nDistribuição teste:")
print(test_df["Label"].value_counts())

## 5. Seleção de features por correlação (calculada só no treino)

In [ ]:
correlations = (
    train_df[numeric_cols]
    .corrwith(train_df["Label_enc"], method="pearson")
    .abs()
    .sort_values(ascending=False)
)

print("\n📊 Top 20 features mais correlacionadas com UDPLag (treino):")
print(correlations.head(20))

plt.figure(figsize=(10, 8))
correlations.head(20).plot(kind="barh")
plt.title("Top 20 Features — Correlação (Pearson) com Label UDPLag (treino)")
plt.xlabel("Correlação Absoluta")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig("/content/feature_correlation_udplag.png", dpi=150)
plt.show()

## 6. Remoção de features redundantes (correlação inter-features)

In [ ]:
N_FEATURES = 5
INTER_CORR_THRESHOLD = 0.90  # 0.85 mais rígido / 0.95 mais permissivo

def select_features_no_redundancy(df_train, ranked, n_features=5, inter_corr_threshold=0.90):
    selected = []
    for feat in ranked.index:
        if len(selected) >= n_features:
            break
        if selected:
            corr_with_selected = df_train[selected + [feat]].corr().loc[feat, selected].abs()
            if (corr_with_selected > inter_corr_threshold).any():
                continue
        selected.append(feat)
    return selected

selected_features = select_features_no_redundancy(
    df_train=train_df,
    ranked=correlations,
    n_features=N_FEATURES,
    inter_corr_threshold=INTER_CORR_THRESHOLD
)

print("\n✅ Features finais (escolhidas no treino):")
for f in selected_features:
    print(f" - {f}: {correlations[f]:.3f}")

## 7. Correlação entre features selecionadas

In [ ]:
corr_between = train_df[selected_features].corr()

print("📊 Correlação entre features selecionadas (treino):")
print(corr_between.round(2))

plt.figure(figsize=(7, 5))
sns.heatmap(corr_between, annot=True, cmap="coolwarm", fmt=".2f", linewidths=0.5)
plt.title("Correlação entre Features Selecionadas — UDPLag (treino)")
plt.tight_layout()
plt.savefig("/content/correlation_between_features_udplag.png", dpi=150)
plt.show()

## 8. Análise de features candidatas (opcional)

In [ ]:
candidates = [
    "Packet Length Mean",
    "URG Flag Count",
    "Down/Up Ratio",
    "Fwd Packets/s",
    "Flow Packets/s",
    "Flow IAT Mean",
    "Flow IAT Std",
    "Bwd Packet Length Mean",
    "Init_Win_bytes_forward"
]

print("📊 Correlação com o label (treino):")
for c in candidates:
    print(f"   {c}: {correlations.get(c, 0):.3f}")

fixed = selected_features[:3] if len(selected_features) >= 3 else selected_features
candidates_in_df = [c for c in candidates if c in train_df.columns]

corr_check = train_df[fixed + candidates_in_df].corr()[fixed].loc[candidates_in_df]
print("\n📊 Correlação das candidatas com as features fixadas (treino):")
print(corr_check.round(2))

## 9. Geração dos datasets JSONL para fine-tuning

In [ ]:
import json
import random

SYSTEM_PROMPT = (
    "You are a network security analyst specialized in detecting UDPLag DDoS attacks. "
    "Analyze network flow features and respond only with a valid JSON object containing "
    "the field 'classification', which must be either 'BENIGN' or 'UDPLag'."
)

USER_TEMPLATES = [
    "Analyze the following network flow and classify it as BENIGN or UDPLag.\n\n"
    "Flow features:\n"
    + "".join([f"- {feat}: {{feat_{i}}}\n" for i, feat in enumerate(selected_features)])
    + '\nRespond ONLY with JSON: {{"classification": "BENIGN" or "UDPLag"}}',

    "Given this network flow data, determine whether it represents a UDPLag flooding attack or normal traffic.\n\n"
    + "".join([f"- {feat}: {{feat_{i}}}\n" for i, feat in enumerate(selected_features)])
    + "\nReturn only valid JSON with the field 'classification'.",
]

def build_user_message(row, feats):
    template = random.choice(USER_TEMPLATES)
    values = {}
    for i, feat in enumerate(feats):
        v = row[feat]
        if isinstance(v, (float, np.floating)):
            v = round(float(v), 2)
        values[f"feat_{i}"] = v
    return template.format(**values)

def make_records(df_split, feats, seed=42):
    random.seed(seed)
    records = []
    for _, row in df_split.iterrows():
        true_label = row["Label"]
        user_msg = build_user_message(row, feats)
        assistant_msg = json.dumps({"classification": true_label})
        records.append({
            "messages": [
                {"role": "system",    "content": SYSTEM_PROMPT},
                {"role": "user",      "content": user_msg},
                {"role": "assistant", "content": assistant_msg},
            ]
        })
    return records

train_records = make_records(train_df.reset_index(drop=True), selected_features, seed=RANDOM_SEED)
test_records  = make_records(test_df.reset_index(drop=True),  selected_features, seed=RANDOM_SEED)

out_train = "/content/finetune_dataset_udplag.jsonl"
out_test  = "/content/test_dataset_udplag.jsonl"

with open(out_train, "w", encoding="utf-8") as f:
    for r in train_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

with open(out_test, "w", encoding="utf-8") as f:
    for r in test_records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("✅ Datasets gerados:")
print(f" - {out_train}  → {len(train_records)} exemplos (treino)")
print(f" - {out_test}   → {len(test_records)} exemplos (teste)")
print(f"\n📊 Distribuição treino: {train_df['Label'].value_counts().to_dict()}")
print(f"📊 Distribuição teste:  {test_df['Label'].value_counts().to_dict()}")
print("\n📋 Exemplo (treino):")
print(json.dumps(train_records[0], indent=2, ensure_ascii=False))